# TexTag: Learning curve from scratch (SmallCNN) vs pretrained (ResNet18)

`08_data_size` showed how the **pretrained** ResNet18 improves with more data. This notebook repeats the experiment with the
**SmallCNN trained from scratch**, on the same slices, and puts both curves on one plot.

**Question:** does a model that starts from scratch need data differently? And roughly how much data is pretraining "worth"?

**Model:** the multi-label SmallCNN from `07_search` (same recipe). **Slices:** the same 5, 10, 20 and 40 images per class as `08`.
**Scores:** test mAP and tag-search precision@10. The ResNet18 numbers are typed in from `08_data_size`.

**Run on Google Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (~35–45 min).

Sections:
1. Setup
2. Data slices
3. Training function
4. Train on each slice
5. Both learning curves
6. How much data is pretraining worth?
7. Findings

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/textag"):
        !git clone -q https://github.com/codecustard/textag.git /content/textag
    %cd /content/textag/notebooks

In [ ]:
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import average_precision_score

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from textag.model import SmallCNN
from textag import multilabel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EPOCHS = os.environ.get("TEXTAG_EPOCHS")   # env var only used for quick test runs

## 2. Data slices

Exactly the slices from `08_data_size`: the first 5, 10, 20 or 40 training images of every class (by main class), nested,
with the full validation and test sets.

In [ ]:
dl = multilabel.loaders(batch_size=32, num_workers=2)
train_ds = dl["train"].dataset
classes = train_ds.classes
test_targets = dl["test"].dataset.targets

main_class = train_ds.dtd._labels          # each training image's main class (0-46)

def slice_indices(per_class):
    """The first `per_class` training images of every class."""
    indices = []
    for c in range(len(classes)):
        in_class = [i for i, label in enumerate(main_class) if label == c]
        indices += in_class[:per_class]
    return indices

SLICES = [5, 10, 20, 40]
for per_class in SLICES:
    print(f"{per_class:>2} per class -> {len(slice_indices(per_class)):>4} training images")

## 3. Training function

The `07_search` SmallCNN recipe: `BCEWithLogitsLoss`, `AdamW` (lr 1e-3, weight decay 1e-4), `ReduceLROnPlateau`,
up to 40 epochs with early stopping (patience 7) on the full validation set. Random starting weights, no pretraining.

In [ ]:
def train_and_score(per_class):
    """Train the multi-label SmallCNN from scratch on one slice; return its test scores."""
    torch.manual_seed(42)
    indices = slice_indices(per_class)
    train_dataloader = DataLoader(Subset(train_ds, indices), batch_size=32, shuffle=True, num_workers=2)

    model = SmallCNN(num_classes=len(classes))
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

    start = timer()
    hist = multilabel.train(model, train_dataloader, dl["val"], optimizer, loss_fn,
                            epochs=int(EPOCHS or 40), device=device, scheduler=scheduler, patience=7, verbose=False)
    seconds = timer() - start

    tag_scores, _ = multilabel.index(model, dl["test"], device)
    return {"per class": per_class,
            "training images": len(indices),
            "test mAP": average_precision_score(test_targets, tag_scores, average="macro"),
            "tag search P@10": multilabel.tag_search_precision(tag_scores, test_targets).mean().item(),
            "epochs run": len(hist["val_loss"]),
            "best epoch": hist["best_epoch"],
            "training time (s)": round(seconds)}

## 4. Train on each slice

Four trainings from scratch, smallest first. The 40-per-class run should land near `07_search`'s SmallCNN (25.1% mAP, 45.7% P@10).

In [ ]:
rows = []
for per_class in SLICES:
    row = train_and_score(per_class)
    rows.append(row)
    print(f"{row['training images']:>4} images: test mAP {row['test mAP']:.1%}, "
          f"tag search P@10 {row['tag search P@10']:.1%}  ({row['training time (s)']} s)")

small = pd.DataFrame(rows).set_index("training images")
small.style.format({"test mAP": "{:.1%}", "tag search P@10": "{:.1%}"})

## 5. Both learning curves

ResNet18 numbers from `08_data_size` (same slices, same metrics). Log-scale x-axis: each step doubles the data.

In [ ]:
resnet = pd.DataFrame({"test mAP": [0.447, 0.538, 0.588, 0.697],
                       "tag search P@10": [0.694, 0.800, 0.834, 0.917]},
                      index=pd.Index([235, 470, 940, 1880], name="training images"))
RANDOM = {"test mAP": None, "tag search P@10": test_targets.mean().item()}   # random P@10 = average share of images with a tag

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, metric in zip(axes, ["test mAP", "tag search P@10"]):
    ax.plot(resnet.index, resnet[metric], "o-", color="#2F5E96", label="ResNet18 (pretrained)")
    ax.plot(small.index, small[metric], "o-", color="#E08A3C", label="SmallCNN (from scratch)")
    for df in (resnet, small):
        for n, value in df[metric].items():
            ax.annotate(f"{value:.0%}", (n, value), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
    if RANDOM[metric] is not None:
        ax.axhline(RANDOM[metric], color="gray", linestyle="--", label="random")
    ax.set_xscale("log")
    ax.set_xticks(resnet.index, [str(n) for n in resnet.index])
    ax.set_xlabel("training images (log scale)")
    ax.set_title(metric)
    ax.set_ylim(0, 1)
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

gains = pd.DataFrame({
    "SmallCNN mAP gain": small["test mAP"].diff(),
    "ResNet18 mAP gain": resnet["test mAP"].diff(),
    "SmallCNN P@10 gain": small["tag search P@10"].diff(),
    "ResNet18 P@10 gain": resnet["tag search P@10"].diff(),
}).dropna()
gains.style.format("{:+.1%}")

## 6. How much data is pretraining worth?

Fit the same **power law** as in `08_data_size` to the SmallCNN points (`error ≈ c × images^k`: each doubling removes the same
share of the remaining error), then solve for how many images SmallCNN would need to match **ResNet18 with only 235 images**.

**This is a rough extrapolation**, often far beyond the measured range: read it as an order of magnitude, not a prediction.

In [ ]:
def fit_power_law(df, metric):
    """error ≈ c × images^k. Returns (c, k)."""
    error = 1 - df[metric]
    k, log_c = np.polyfit(np.log(df.index), np.log(error), 1)
    return np.exp(log_c), k

def readable(x, unit=""):
    """Big or impossible estimates as words instead of 30-digit numbers."""
    if not np.isfinite(x):
        return "no estimate"
    if x > 1e9:
        return "over 1 billion" + unit
    return f"{x:,.0f}{unit}"

rows = []
for metric in ["test mAP", "tag search P@10"]:
    c, k = fit_power_law(small, metric)
    target = resnet[metric].iloc[0]                       # ResNet18's score with 235 images
    if k < 0:
        needed = ((1 - target) / c) ** (1 / k)            # solve 1 - c × n^k = target for n
    else:
        needed = float("inf")                             # SmallCNN didn't improve: no estimate
    rows.append({"metric": metric,
                 "SmallCNN error per doubling": f"× {2 ** k:.2f}",
                 "ResNet18 with 235 images": f"{target:.1%}",
                 "SmallCNN images needed (est.)": readable(needed),
                 "≈ times more data": readable(needed / 235, "×")})
pd.DataFrame(rows).set_index("metric")

## 7. Findings

*(Filled in after the full run.)*